In [ ]:
import transformers
import trl
import peft

In [ ]:
import torch
from transformers import (AutoModelForCausalLM,BitsAndBytesConfig,AutoTokenizer,DataCollatorForLanguageModeling,TrainingArguments,
                          EarlyStoppingCallback,pipeline)
from datasets import load_dataset
from peft import LoraConfig,get_peft_model
from trl import SFTTrainer

In [ ]:
dataset=load_dataset("gbharti/finance-alpaca")
dataset = dataset["train"].select(range(5000))

In [ ]:
def chat_format(example):
    text=f"""<s>[INST]{example['instruction']}[/INST]{example['output']}</s>"""
    return {"text": text}
dataset=dataset.map(chat_format)

In [ ]:
dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

In [ ]:
checkpoint="TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
bnb=BitsAndBytesConfig(load_in_4bit=True,
                      bnb_4bit_quant_type="nf4",
                      bnb_4bit_compute_dtype=torch.float16)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(checkpoint)
tokenizer.pad_token=tokenizer.eos_token

In [ ]:
model=AutoModelForCausalLM.from_pretrained(checkpoint,
                           quantization_config=bnb,
                          device_map="auto")

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"],max_length=256,truncation=True)
tokenized_dataset=dataset.map(tokenize_function)

In [ ]:
DataCollator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

In [ ]:
lora_config=LoraConfig(r=8,
                       lora_alpha=16,
                       target_modules=["q_proj","v_proj"],
                       bias="none",
                       lora_dropout=0.05,
                       task_type="CAUSAL_LM")

In [ ]:
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

In [ ]:
training_args=TrainingArguments(output_dir="finance_chat_model",
                               per_device_train_batch_size=2,
                               per_device_eval_batch_size=2,
                               num_train_epochs=2,
                               learning_rate=2e-4,
                               eval_strategy="epoch",
                               save_strategy="epoch",
                               load_best_model_at_end=True,
                               fp16=True,
                               report_to="none") 

In [ ]:
trainer=SFTTrainer(model=model,
                   train_dataset=tokenized_dataset["train"],
                   eval_dataset=tokenized_dataset["test"],
                   data_collator=DataCollator,
                   args=training_args)

In [ ]:
trainer.train()

In [ ]:
eval_results=trainer.evaluate()
with open("evaluation.txt", "w") as f:
    f.write(f"Evaluation Results\n")
    f.write(f"==================\n\n")

    for key, value in eval_results.items():
        f.write(f"{key}: {value}\n")

In [ ]:
model.save_pretrained("finance_chat_model")
tokenizer.save_pretrained("finance_chat_model")